In [ ]:
pip install fastf1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.6/148.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.0/165.0 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 15.0.1
    Uninstalling websockets-15.0.1:
      Successfully uninstalled websockets-15.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.14.1 requires websockets<16.0.0,>=15.0.1, but you have websockets 13.1 which is incompatible.
dataproc-spark-connect 0.8.3 requires websockets>=14.0, but you have websockets 13.1 which is incompatible.


In [ ]:
import os
import fastf1

# Step 1: Create folder if it doesn't exist
os.makedirs('/content/fastf1_cache', exist_ok=True)

# Step 2: Enable caching (now it will work)
fastf1.Cache.enable_cache('/content/fastf1_cache')


In [ ]:
# --- 2) (Optional) check cache is active ---
if fastf1.Cache._CACHE_DIR is not None:
  print("FastF1 caching is enabled.")
else:
  print("FastF1 caching is not enabled.")

FastF1 caching is enabled.


In [ ]:
# Copy-paste this entire cell into Colab and run.

import datetime
import sys
import pandas as pd

# --- 0) Import fastf1 and check installation ---
try:
    import fastf1
except Exception as e:
    print("ERROR: fastf1 import failed. You may need to restart the runtime (Runtime -> Restart runtime).")
    print("Exception:", e)
    raise

print("fastf1 version:", fastf1.__version__)

# Optional: if you saw the websockets / dependency warnings, a runtime restart is sometimes needed.
print("If you saw dependency conflicts during pip install, and fastf1 import failed, restart Colab runtime and re-run this cell.")

# --- 1) enable cache (fastf1 caching speeds repeated runs) ---
fastf1.Cache.enable_cache('/content/fastf1_cache')

# --- 2) get schedule for the current year and pick the latest event that has already happened ---
year = datetime.datetime.now().year
print("Using year:", year)

try:
    schedule = fastf1.get_event_schedule(year)
except Exception as e:
    # fallback: try previous year if current year schedule not yet available
    print("Could not fetch schedule for", year, "-> trying previous year. Error:", e)
    year = year - 1
    schedule = fastf1.get_event_schedule(year)

# convert EventDate to datetime if needed and pick latest date <= today
if 'EventDate' in schedule.columns:
    schedule['EventDate'] = pd.to_datetime(schedule['EventDate'])
    today = pd.to_datetime(datetime.date.today())
    past_events = schedule[schedule['EventDate'] <= today].copy()
    if past_events.empty:
        # if none, just pick last event in schedule
        chosen = schedule.iloc[-1]
    else:
        chosen = past_events.iloc[-1]
else:
    # if EventDate missing, pick last row
    chosen = schedule.iloc[-1]

event_name = chosen['EventName'] if 'EventName' in chosen.index else chosen.get('EventName', None)
round_number = chosen.get('Round', None)
print(f"Selected event -> Round: {round_number}, EventName: {event_name}, EventDate: {chosen.get('EventDate', 'N/A')}")

# --- 3) try to load the Race session for that event ---
# fastf1 expects the event parameter as the event name or GP name; sometimes exact string differs.
# We'll try a few variants to maximize success.
possible_names = [event_name, f"{event_name}", str(chosen.get('GP', '')), str(chosen.get('Event', ''))]
# remove None and duplicates
possible_names = [p for p in pd.unique(possible_names) if isinstance(p, str) and p.strip()!='']

loaded = False
errors = []
for name in possible_names:
    try:
        print("Trying to load session for:", name)
        session = fastf1.get_session(year, name, 'R')  # 'R' = Race
        session.load()
        print("Loaded session for:", name)
        loaded = True
        break
    except Exception as e:
        errors.append((name, str(e)))
        print("Failed for:", name, "->", e)

if not loaded:
    # fallback: try to find the event by looping schedule and trying names
    print("Automatic name attempts failed. Trying all schedule event names...")
    for idx, row in schedule.iterrows():
        try:
            nm = row.get('EventName', None) or row.get('GP', None) or row.get('Event', None)
            if not nm:
                continue
            session = fastf1.get_session(year, nm, 'R')
            session.load()
            event_name = nm
            print("Loaded session with fallback name:", nm)
            loaded = True
            break
        except Exception:
            continue

if not loaded:
    print("Could not load the Race session automatically. See attempts and errors below:")
    for nm, err in errors:
        print(f" - {nm}: {err}")
    raise RuntimeError("Failed to load race session. You can try specifying the event name manually.")

# --- 4) extract laps, convert times, basic cleaning, save CSV ---
laps = session.laps.copy()

# show available columns
print("\nLaps columns:", list(laps.columns))

# convert LapTime to seconds if present
if 'LapTime' in laps.columns:
    laps['LapTime_s'] = laps['LapTime'].dt.total_seconds()
else:
    print("Warning: 'LapTime' column not found in laps. Inspect laps dataframe manually.")

# Ensure pit flags exist
for col in ['IsPitInLap', 'IsPitOutLap']:
    if col not in laps.columns:
        laps[col] = False

# Basic cleaning: remove pit in/out laps and NaN LapTime
clean = laps[~(laps['IsPitInLap'] | laps['IsPitOutLap'])]
if 'LapTime_s' in clean.columns:
    clean = clean[clean['LapTime_s'].notna()]

# Save cleaned CSV
out_fname = f"latest_race_clean_{year}_{event_name.replace(' ', '_')}.csv"
clean.to_csv(out_fname, index=False)
print(f"\n✅ Saved cleaned laps to: {out_fname}")
print("Clean rows:", len(clean))

# show a small sample
display_cols = ['Driver','LapNumber','LapTime_s','Compound','TyreLife','IsPitInLap','IsPitOutLap']
print("\nSample rows:")
display(clean[display_cols].head(15))


fastf1 version: 3.6.1
If you saw dependency conflicts during pip install, and fastf1 import failed, restart Colab runtime and re-run this cell.
Using year: 2025
Selected event -> Round: None, EventName: Singapore Grand Prix, EventDate: 2025-10-05 00:00:00


/tmp/ipython-input-1474360644.py:58: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  possible_names = [p for p in pd.unique(possible_names) if isinstance(p, str) and p.strip()!='']
core           INFO 	Loading data for Singapore Grand Prix - Race [v3.6.1]
INFO:fastf1.fastf1.core:Loading data for Singapore Grand Prix - Race [v3.6.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...


Trying to load session for: Singapore Grand Prix


req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
INFO:fastf1.api:Fetching session status data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
INFO:fastf1.fastf1.req:No cached data found for lap_count. Loading data...
_api           INFO 	F

Loaded session for: Singapore Grand Prix

Laps columns: ['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate']

✅ Saved cleaned laps to: latest_race_clean_2025_Singapore_Grand_Prix.csv
Clean rows: 1229

Sample rows:


,Driver,LapNumber,LapTime_s,Compound,TyreLife,IsPitInLap,IsPitOutLap
0,RUS,1.0,103.905,MEDIUM,1.0,False,False
1,RUS,2.0,99.149,MEDIUM,2.0,False,False
2,RUS,3.0,98.035,MEDIUM,3.0,False,False
3,RUS,4.0,97.570,MEDIUM,4.0,False,False
4,RUS,5.0,97.184,MEDIUM,5.0,False,False
5,RUS,6.0,96.854,MEDIUM,6.0,False,False
6,RUS,7.0,96.768,MEDIUM,7.0,False,False
7,RUS,8.0,96.655,MEDIUM,8.0,False,False
8,RUS,9.0,96.726,MEDIUM,9.0,False,False
9,RUS,10.0,96.665,MEDIUM,10.0,False,False


In [ ]:


import os, datetime, pandas as pd
import fastf1
from google.colab import files

# 0) Create cache folder (only once)
os.makedirs('/content/fastf1_cache', exist_ok=True)
fastf1.Cache.enable_cache('/content/fastf1_cache')

print("fastf1 version:", fastf1.__version__)

# 1) Choose year and pick latest past event automatically
year = datetime.datetime.now().year
schedule = fastf1.get_event_schedule(year)
schedule['EventDate'] = pd.to_datetime(schedule['EventDate'])
today = pd.to_datetime(datetime.date.today())
past_events = schedule[schedule['EventDate'] <= today].copy()
if past_events.empty:
    chosen = schedule.iloc[-1]
else:
    chosen = past_events.iloc[-1]

event_name = chosen.get('EventName') or chosen.get('GP') or chosen.get('Event')
round_num = chosen.get('Round') or chosen.get('RoundNumber')
print(f"Selected event -> Round: {round_num}, EventName: {event_name}, EventDate: {chosen.get('EventDate')}")

# 2) Load Race session (try multiple possible names if needed)
tried = []
loaded = False
candidates = [event_name, str(chosen.get('GP','')), str(chosen.get('Event',''))]
for name in pd.unique(candidates):
    if not isinstance(name, str) or not name.strip():
        continue
    try:
        print("Trying:", name)
        session = fastf1.get_session(year, name, 'R')
        session.load()
        print("Loaded session:", name)
        loaded = True
        event_name = name
        break
    except Exception as e:
        tried.append((name, str(e)))
        print("Failed:", name, "->", e)

if not loaded:
    raise RuntimeError("Failed to load race session. Try specifying event name manually.")

# 3) Save laps table (this is the CSV you can open in Excel)
laps = session.laps.copy()
# convert LapTime timedelta to seconds (helpful in Excel)
if 'LapTime' in laps.columns:
    laps['LapTime_s'] = laps['LapTime'].dt.total_seconds()
# filename (safe)
safe_name = event_name.replace(' ', '_').replace('/', '_')
laps_fname = f"laps_{year}_{safe_name}.csv"
laps.to_csv(laps_fname, index=False)
print("Saved laps CSV:", laps_fname, " (open in Excel)")

# OPTIONAL 4) If you want telemetry CSVs: (can be large). Uncomment to run.
# This will create a folder /content/telemetry_<event> and save one csv per lap (driver + lap number).
save_telemetry = False   # <-- set True if you want telemetry files (large)
if save_telemetry:
    telem_dir = f"/content/telemetry_{year}_{safe_name}"
    os.makedirs(telem_dir, exist_ok=True)
    print("Saving telemetry per lap into:", telem_dir)
    # iterate through laps and save telemetry for each
    for idx, lap in laps.iterrows():
        try:
            # pick lap by session.laps.loc[...] can use lap['LapNumber'] and lap['Driver']
            # safe selection: use session.laps.pick_* helpers
            dcode = lap.get('Driver', None)
            lapnum = int(lap.get('LapNumber', -1))
            # pick specific lap and get telemetry
            lap_row = session.laps.pick_driver(dcode).loc[session.laps.pick_driver(dcode)['LapNumber'] == lapnum]
            if lap_row.shape[0] == 0:
                continue
            lap_obj = lap_row.iloc[0]
            tel = lap_obj.get_telemetry()
            if tel is None or tel.empty:
                continue
            tel_fname = os.path.join(telem_dir, f"telemetry_{dcode}_lap{lapnum}.csv")
            tel.to_csv(tel_fname, index=False)
        except Exception as e:
            # skip errors to avoid stopping long runs
            print("Telemetry save error for", dcode, "lap", lapnum, "->", e)
    # zip telemetry folder for download
    zipname = f"telemetry_{year}_{safe_name}.zip"
    !zip -r -q $zipname $telem_dir
    print("Zipped telemetry to:", zipname)

# 5) Provide download links / convenience: download laps CSV now
print("\nYou can download the laps CSV file now. In Colab left Files pane, right-click -> Download.")
# also provide programmatic download
files.download(laps_fname)

# If telemetry saved and zipped, it will be available for download above as well.
# ---------- END CELL ----------


/tmp/ipython-input-1200480052.py:30: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  for name in pd.unique(candidates):
core           INFO 	Loading data for Singapore Grand Prix - Race [v3.6.1]
INFO:fastf1.fastf1.core:Loading data for Singapore Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info
INFO:fastf1.fastf1.req:Using cached data for session_info
req            INFO 	Using cached data for driver_info
INFO:fastf1.fastf1.req:Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
INFO:fastf1.fastf1.req:Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
INFO:fastf1.fastf1.req:Using cached data for lap_count
req            INFO 	Using cached data for track_status_data


fastf1 version: 3.6.1
Selected event -> Round: 18, EventName: Singapore Grand Prix, EventDate: 2025-10-05 00:00:00
Trying: Singapore Grand Prix


INFO:fastf1.fastf1.req:Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
INFO:fastf1.fastf1.req:Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
INFO:fastf1.fastf1.req:Using cached data for timing_app_data
core           INFO 	Processing timing data...
INFO:fastf1.fastf1.core:Processing timing data...
req            INFO 	Using cached data for car_data
INFO:fastf1.fastf1.req:Using cached data for car_data
req            INFO 	Using cached data for position_data
INFO:fastf1.fastf1.req:Using cached data for position_data
req            INFO 	Using cached data for weather_data
INFO:fastf1.fastf1.req:Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
INFO:fastf1.fastf1.req:Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['63', '1', '4', '81', '12', '16', '14', '44', '87', '55', '6

Loaded session: Singapore Grand Prix
Saved laps CSV: laps_2025_Singapore_Grand_Prix.csv  (open in Excel)

You can download the laps CSV file now. In Colab left Files pane, right-click -> Download.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>